In [1]:

import os, random, unicodedata
from hashlib import blake2b
from collections import Counter


In [2]:

ROOT = "D:/Tasks/Project_SLM"
NGRAM, BANDS, ROWS = 5, 8, 4
MIN_CHARS, MAX_CHARS = 200, 100_000
MASK = (1 << 61) - 1
random.seed(0)
PERM = [(random.getrandbits(60) | 1, random.getrandbits(60)) for _ in range(BANDS * ROWS)]


In [3]:

# every Telugu raw file, oldest first (first-seen wins on duplicates)
TE_FILES = [f"{ROOT}/raw/te2.txt"]


def signature(doc):
    w = doc.split()
    grams = {" ".join(w[i:i+NGRAM]) for i in range(len(w)-NGRAM+1)} or {doc}
    base = [int.from_bytes(blake2b(g.encode(), digest_size=8).digest(), "big") & MASK
            for g in grams]
    mins = [min((a*h+b) & MASK for h in base) for a, b in PERM]
    return [b"".join(m.to_bytes(8, "big") for m in mins[i*ROWS:(i+1)*ROWS])
            for i in range(BANDS)]


def is_telugu(doc, min_ratio=0.70):
    letters = [c for c in doc if c.isalpha()]
    if not letters:
        return False
    return sum("\u0c00" <= c <= "\u0c7f" for c in letters) / len(letters) >= min_ratio


def clean_te():
    seen, bands, s = set(), [set() for _ in range(BANDS)], Counter()
    per_file = {}
    os.makedirs(f"{ROOT}/clean", exist_ok=True)
    with open(f"{ROOT}/clean/te.txt", "w", encoding="utf-8") as fout:
        for path in TE_FILES:
            if not os.path.exists(path):
                print(f"--- {path}  MISSING, skipped"); continue
            print(f"--- {path}", flush=True)
            before_kept, before_bytes = s["kept"], s["bytes"]
            with open(path, encoding="utf-8") as fin:
                for line in fin:
                    s["in"] += 1
                    doc = unicodedata.normalize("NFC", line.strip())
                    if not (MIN_CHARS <= len(doc) <= MAX_CHARS):
                        s["drop_len"] += 1; continue
                    h = blake2b(doc.encode(), digest_size=16).digest()
                    if h in seen:
                        s["drop_exact"] += 1; continue
                    seen.add(h)
                    if not is_telugu(doc):
                        s["drop_script"] += 1; continue
                    sig = signature(doc)
                    if sum(b in bands[i] for i, b in enumerate(sig)) >= 2:
                        s["drop_near"] += 1; continue
                    for i, b in enumerate(sig):
                        bands[i].add(b)
                    fout.write(doc + "\n")
                    s["kept"] += 1; s["bytes"] += len(doc.encode())
                    if s["in"] % 200_000 == 0:
                        print(f"  {s['in']:,} read, {s['kept']:,} kept, "
                              f"{s['bytes']/1e9:.2f} GB", flush=True)
            per_file[path] = (s["kept"] - before_kept, s["bytes"] - before_bytes)

    print(f"\nte: in={s['in']:,} kept={s['kept']:,} "
          f"({s['kept']/max(s['in'],1):.1%}) {s['bytes']/1e9:.2f} GB")
    for k in ("drop_len", "drop_exact", "drop_script", "drop_near"):
        print(f"   {k:<12}{s[k]:>12,}")
    print("\ncontribution per source (unique docs surviving dedup):")
    for p, (k, b) in per_file.items():
        print(f"   {os.path.basename(p):<20}{k:>10,} docs  {b/1e9:>6.2f} GB")

    # ---- the decision numbers ----
    tok = s["bytes"] / 11.03          # o200k_pat fertility, measured
    print(f"\nestimated Telugu tokens (o200k): {tok/1e9:.2f}B")
    print(f"=> total budget at 25% share:    {tok/0.25/1e9:.2f}B tokens")
    print(f"=> Chinchilla-optimal model:     {tok/0.25/20/1e6:.0f}M params")
    print(f"\nen needed: {tok/0.25*0.40/1e9:.2f}B   (have 3.71B)")
    print(f"de needed: {tok/0.25*0.35/1e9:.2f}B   (have 3.09B)")
    return s


if __name__ == "__main__":
    clean_te()

--- C:/Project_SLM/raw/te2.txt
  200,000 read, 192,692 kept, 1.75 GB
  400,000 read, 384,922 kept, 3.37 GB
  600,000 read, 576,637 kept, 4.92 GB
  800,000 read, 767,725 kept, 6.44 GB
  1,000,000 read, 958,282 kept, 7.90 GB
  1,200,000 read, 1,147,925 kept, 9.33 GB
  1,400,000 read, 1,336,973 kept, 10.75 GB
  1,600,000 read, 1,525,229 kept, 12.14 GB
  1,800,000 read, 1,712,602 kept, 13.52 GB
  2,000,000 read, 1,898,014 kept, 14.81 GB
  2,200,000 read, 2,078,292 kept, 15.75 GB
  2,400,000 read, 2,258,462 kept, 16.70 GB
  2,600,000 read, 2,438,552 kept, 17.64 GB
  2,800,000 read, 2,618,574 kept, 18.59 GB
  3,200,000 read, 2,978,379 kept, 20.48 GB
  3,400,000 read, 3,158,053 kept, 21.42 GB
  3,800,000 read, 3,516,979 kept, 23.30 GB
  4,200,000 read, 3,876,000 kept, 25.19 GB

te: in=4,346,994 kept=4,007,683 (92.2%) 25.89 GB
   drop_len             504
   drop_exact         9,969
   drop_script      114,803
   drop_near        214,035

contribution per source (unique docs surviving dedup):
 